# Phase 1 — Batch Trainer

Dhaka PM2.5 one-hour-ahead forecasting. Trains a `GBTRegressor` on the historical
AirNow series and saves the `PipelineModel` that Phase 2's streaming job loads.

**No streaming in this notebook.** Phase 1 is pure batch. Finish it — get an
honest RMSE that beats the persistence baseline — before touching Phase 2.

### Runtime

Pick a **CPU** runtime. Spark MLlib is CPU-only; `GBTRegressor` has no CUDA path,
so a GPU runtime spends compute units for zero speedup.

### Editing workflow

This notebook is a thin driver. All logic lives in `src/*.py` in the repo.
When the code changes, re-run the **Setup** cell below — `git pull` plus
`autoreload` pick it up with no re-upload and no kernel restart.

## 0 · Setup

**Re-run this cell whenever the code is updated.** It pulls the latest
`src/*.py` and hot-reloads them into the running kernel.

In [ ]:
# ==========================================================
#   SETUP  --  RE-RUN THIS CELL EVERY TIME THE CODE UPDATES
#   Pulls the latest src/ from GitHub. Nothing to upload.
# ==========================================================
import os

REPO = "https://github.com/mashrur-rahman-fahim/DA-Dhaka-air-quality-prediction.git"
BASE = "/content" if os.path.isdir("/content") else os.path.expanduser("~")
REPO_DIR = os.path.join(BASE, "DA-Dhaka-air-quality-prediction")

!pip install -q pyspark==3.5.1

if not os.path.isdir(REPO_DIR + "/.git"):
    !git clone -q {REPO} {REPO_DIR}

os.chdir(REPO_DIR)

!git pull
!git log --oneline -1

In [ ]:
# autoreload swaps in the freshly pulled src/*.py with no runtime restart.
# reload_ext (not load_ext) so re-running this cell stays warning-free.
%reload_ext autoreload
%autoreload 2

from src.session import get_spark
from src import config, data, features, train

print("modules loaded")

## 1 · Get the data

Public AirNow embassy archive on S3 — no API key, no manual upload.
Ten yearly CSVs, ~8 MB total, covering 2016-01-01 to 2025-03-24.

`dosairnowdata.org`, the URL in most tutorials and in the old README, is dead.
This bucket is the live mirror the EPA embassy map itself reads from.

In [ ]:
data.download_raw()

## 2 · Spark session

In [ ]:
spark = get_spark()
spark.sparkContext.setLogLevel("ERROR")

print("Spark", spark.version)

## 3 · Load with an explicit schema

Never `inferSchema`. The schema is written once in `src/schema.py` and reused by
Phase 2, where `spark.readStream` **refuses** to infer one.

In [ ]:
raw = data.load_raw()

print(f"raw rows: {raw.count():,}")
raw.show(5, truncate=False)

## 4 · Clean

The QC filter is **not** a no-op on the real archive — 2,342 of 77,710 rows are
`Missing` / `Invalid` / `Suspect` and carry a `-999` sentinel. Feed those to the
model and it learns garbage.

In [ ]:
raw.groupBy("qc").count().orderBy("count", ascending=False).show()

In [ ]:
clean = data.clean(raw).cache()

print(f"valid rows: {clean.count():,}")
clean.select("ts", "pm25", "nowcast", "AQI", "aqi_category").show(5)
clean.select("pm25").describe().show()

## 5 · Gap-safe hourly spine

**The step that makes every lag feature correct.**

`F.lag()` counts *rows*, not hours. The series is missing thousands of hours
across hundreds of gaps, so lagging raw rows would hand the model a reading from
days earlier while labelling it `lag_1`. Generating every hour in the range and
left-joining puts an explicit null in each hole instead.

In [ ]:
spine = data.hourly_spine(spark, clean).cache()

n_spine, n_valid = spine.count(), clean.count()
print(f"spine hours : {n_spine:,}")
print(f"real rows   : {n_valid:,}")
print(f"holes       : {n_spine - n_valid:,}  ({(n_spine - n_valid) / n_spine:.1%})")

## 6 · Features

All built strictly from hours before *t*, so this is a genuine one-hour-ahead
forecast.

Leakage rules enforced in `src/features.py`:

- rolling windows end at `-1`, never `0`
- `NowCast` is used only lagged — at time *t* it is a smoothed function of
  readings that **include** *t*, i.e. the answer

In [ ]:
feat = features.drop_incomplete(features.build_features(spine)).cache()

print(f"usable rows: {feat.count():,}")
print(f"features ({len(config.FEATURE_COLS)}): {config.FEATURE_COLS}")
feat.select("ts", "pm25", "lag_1", "lag_24", "roll_mean_3", "roll_std_24").show(5)

## 7 · Time split

Never `randomSplit`. Shuffling rows lets the model train on the future and test
on the past, which inflates every metric into meaninglessness.

In [ ]:
tr, te = train.time_split(feat)
tr.cache()
te.cache()

print(f"split at   : {config.SPLIT_TS}")
print(f"train rows : {tr.count():,}")
print(f"test rows  : {te.count():,}")

tr.agg({"ts": "max"}).show()
te.agg({"ts": "min"}).show()

## 8 · Persistence baseline

*"Next hour = this hour."* The bar. A model that cannot beat this has learned
nothing about the series, no matter how good its RMSE looks in isolation.

In [ ]:
baseline_rmse = train.persistence_rmse(te)

print(f"persistence RMSE: {baseline_rmse:.3f} ug/m3")

## 9 · Train

In [ ]:
import time

pipeline = train.build_pipeline()

t0 = time.time()
model = pipeline.fit(tr)
print(f"trained in {time.time() - t0:.1f}s")

## 10 · Evaluate

In [ ]:
metrics = train.evaluate(model, te)
lift = (baseline_rmse - metrics["rmse"]) / baseline_rmse

print(f"RMSE : {metrics['rmse']:.3f} ug/m3")
print(f"MAE  : {metrics['mae']:.3f} ug/m3")
print(f"R2   : {metrics['r2']:.4f}")
print()
print(f"baseline RMSE : {baseline_rmse:.3f}")
print(f"improvement   : {lift:+.1%}")
print()
print("VERDICT:", "beats baseline" if metrics["rmse"] < baseline_rmse
      else "DOES NOT beat baseline -- do not proceed to Phase 2")

In [ ]:
for name, imp in train.feature_importances(model):
    if imp > 0.001:
        print(f"{name:>16}  {imp:.4f}  {'#' * int(imp * 120)}")

## 11 · Look at the predictions

In [ ]:
import matplotlib.pyplot as plt

pdf = (model.transform(te)
       .select("ts", "pm25", "prediction")
       .orderBy("ts")
       .limit(24 * 21)          # first three weeks of the test period
       .toPandas())

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(pdf["ts"], pdf["pm25"], label="actual", lw=1.2)
ax.plot(pdf["ts"], pdf["prediction"], label="predicted", lw=1.2, alpha=0.85)
ax.set_ylabel("PM2.5 (ug/m3)")
ax.set_title("Test period - actual vs one-hour-ahead forecast")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 12 · Save the model

Save the whole `PipelineModel`, not the bare `GBTRegressor` — the
`VectorAssembler` stage has to travel with it, or the streaming job in Phase 2
cannot build a `features` vector.

This directory is the entire handoff between phases.

In [ ]:
path = train.save(model)
print("saved ->", path)

### Keep the model

Colab wipes local disk when the runtime dies, and `models/pm25_v1` is what
Phase 2 loads. Run the cell below to copy it to Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/dhaka_pm25
!cp -r models /content/drive/MyDrive/dhaka_pm25/
!ls -R /content/drive/MyDrive/dhaka_pm25 | head -20

## Next

1. Read the RMSE line above. If it does not beat the persistence baseline,
   stop — tune features, do not start Phase 2.
2. `models/pm25_v1/` is the artifact Phase 2 requires in order to exist.
3. Phase 2 (`readStream` → `foreachBatch` → `PipelineModel.load` → `transform`)
   reuses `src/features.py` unchanged, so the two phases cannot drift apart.